In [ ]:
import os
import pandas as pd
from boxsdk import Client, CCGAuth
from dotenv import load_dotenv

load_dotenv(override=True)

CLIENT_ID = os.getenv("BOXCLIENTID")
CLIENT_SECRET = os.getenv("BOXCLIENTSECRET")
assert CLIENT_ID and CLIENT_SECRET, "Set BOXCLIENTID and BOXCLIENTSECRET in your environment"

auth = CCGAuth(client_id=CLIENT_ID, client_secret=CLIENT_SECRET, enterprise_id=20888)
client = Client(auth)

In [ ]:
FOLDER_ID = "370739222165"  # EU-ANALYTICAL-CUSTOMER-OUTPUT


def list_folder_items(client, folder_id, limit=1000):
    folder = client.folder(folder_id).get()
    items = []
    offset = 0
    while True:
        batch = list(
            client.folder(folder_id).get_items(
                limit=limit,
                offset=offset,
                fields=["id", "name", "type", "size", "modified_at", "created_at"],
            )
        )
        if not batch:
            break
        items.extend(batch)
        if len(batch) < limit:
            break
        offset += limit
    return folder, items


folder, items = list_folder_items(client, FOLDER_ID)
print(f"Folder: {folder.name} (id={folder.id})")
print(f"Total items: {len(items)}")

In [ ]:
rows = [
    {
        "type": item.type,
        "id": item.id,
        "name": item.name,
        "size": getattr(item, "size", None),
        "modified_at": getattr(item, "modified_at", None),
        "created_at": getattr(item, "created_at", None),
    }
    for item in sorted(items, key=lambda x: (x.type, x.name.lower()))
]

folder_df = pd.DataFrame(rows)
folder_df